In [1]:
import scanpy as sc
import anndata as an
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Union
import seaborn as sns
import scipy
import bbknn
from scipy.io import mmread

/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import scib
from scib.metrics import silhouette_batch
from scib.metrics import kBET
from scib.metrics import ilisi_graph
from scib.metrics import pcr_comparison
from scib.metrics import metrics_all

In [11]:
import random

In [3]:
from scipy import sparse

In [4]:
import os
os.environ['R_HOME'] = '/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/R'
os.environ['R_USER'] = os.environ['R_HOME']
np.float_ = np.float64

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# E12

In [6]:
cyto = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E12f_CytO_new_clustering.h5ad')
spla = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E12f_splatter.h5ad')
simp = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E12f_splatter_simple.h5ad')
scde = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E12f_scDesign3.h5ad')
scva = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E12f_scVAEDer.h5ad')
real = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/data_standard/Zygote_to_CS7.h5ad')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
real = real[real.obs.time == 'E12']
real.var = real.var.rename(columns = {'x':'features'})
real.var.index = real.var.features

In [8]:
set(real.obs.merge_type)

{'Epiblast', 'Hypoblast', 'Others', 'Trophoblast', 'YolkSac'}

In [15]:
adata_int = sc.concat([cyto,spla,simp,scde,scva],
                      join = 'outer',
                      fill_value = 0)

adata = sc.concat([real,adata_int],
                 label = 'source',
                 keys = ['real','simulated'],
                 index_unique = '-')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
set_seed(seed=42)
adata.obs['stage'] = 'E12'
sc.tl.pca(adata, svd_solver='arpack')
sc.external.pp.bbknn(adata, batch_key='source', pynndescent_random_state=42)
sc.tl.umap(adata)

In [17]:
adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')

/tmp/ipykernel_78831/2877064320.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')


In [22]:
kbet_score_cyto = kBET(
    adata[adata.obs.time.isin(['E12','E12-CytOrigin'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scde = kBET(
    adata[adata.obs.time.isin(['E12','E12-scDesign3'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scva = kBET(
    adata[adata.obs.time.isin(['E12','E12-scVAEDer'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_spla = kBET(
    adata[adata.obs.time.isin(['E12','E12-Splatter'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_simp = kBET(
    adata[adata.obs.time.isin(['E12','E12-Splatter-Simple'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pand

0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/

1 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of o

...

/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


Adding diffusion to step 4


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


4 labels consist of a single batch or is too small. Skip.
4 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(


In [ ]:
dfs = {
    'CytOrigin': kbet_score_cyto,
    'scVAEDer': kbet_score_scva,
    'scDesign3': kbet_score_scde,
    'Splatter': kbet_score_spla,
    'Splatter-Simple': kbet_score_simp
}
combined = pd.concat([df.set_index('cluster')['kBET'].rename(name) for name, df in dfs.items()], axis=1)
combined.fillna(0, inplace=True)
combined.loc['mean'] = combined.mean(axis=0)
combined.T.to_csv('kBET/E12_full_pca.csv')

# E14

In [25]:
cyto = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E14f_CytO_new_clustering.h5ad')
spla = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E14f_splatter.h5ad')
simp = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E14f_splatter_simple.h5ad')
scde = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E14f_scDesign3.h5ad')
scva = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/E14f_scVAEDer.h5ad')
real = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/data_standard/Zygote_to_CS7.h5ad')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
real = real[real.obs.time == 'E14']
real.var = real.var.rename(columns = {'x':'features'})
real.var.index = real.var.features

In [27]:
adata_int = sc.concat([cyto,scde,spla,simp,scva],
                      join = 'outer',
                      fill_value = 0)

adata = sc.concat([real,adata_int],
                 label = 'source',
                 keys = ['real','simulated'],
                 index_unique = '-')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [28]:
set_seed(seed=42)
sc.tl.pca(adata, svd_solver='arpack')
adata.obs['stage'] = 'E14'
sc.external.pp.bbknn(adata, batch_key='source', pynndescent_random_state=42)
sc.tl.umap(adata)

In [29]:
adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')

/tmp/ipykernel_78831/2877064320.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')


In [34]:
kbet_score_cyto = kBET(
    adata[adata.obs.time.isin(['E14','E14-CytOrigin'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scde = kBET(
    adata[adata.obs.time.isin(['E14','E14-scDesign3'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scva = kBET(
    adata[adata.obs.time.isin(['E14','E14-scVAEDer'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_spla = kBET(
    adata[adata.obs.time.isin(['E14','E14-Splatter'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_simp = kBET(
    adata[adata.obs.time.isin(['E14','E14-Splatter-Simple'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pand

0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a 

0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_cou

0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_cou

4 labels consist of a single batch or is too small. Skip.
4 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(


In [ ]:
dfs = {
    'CytOrigin': kbet_score_cyto,
    'scVAEDer': kbet_score_scva,
    'scDesign3': kbet_score_scde,
    'Splatter': kbet_score_spla,
    'Splatter-Simple': kbet_score_simp
}
combined = pd.concat([df.set_index('cluster')['kBET'].rename(name) for name, df in dfs.items()], axis=1)
combined.fillna(0, inplace=True)
combined.loc['mean'] = combined.mean(axis=0)
combined.T.to_csv('kBET/E14_full_pca.csv')

# CS10

In [37]:
cyto = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS10f_CytO_new_new_clustering.h5ad')
spla = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS10f_splatter.h5ad')
simp = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS10f_splatter_simple.h5ad')
scde = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS10f_scDesign3.h5ad')
scva = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS10f_scVAEDer.h5ad')
real = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/data_standard/CS10.h5ad')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [38]:
real.var = real.var.rename(columns = {'x':'features'})
real.var.index = real.var.features
real.obs['time'] = 'CS10'

In [39]:
adata_int = sc.concat([cyto,scva,spla,simp,scde], join = 'outer', fill_value = 0)

adata = sc.concat([real,adata_int], label = 'source', keys = ['real','simulated'], index_unique = '-')

set_seed(seed=42)
sc.tl.pca(adata, svd_solver='arpack')
adata.obs['stage'] = 'CS10'
sc.external.pp.bbknn(adata, batch_key='source', pynndescent_random_state=42)
sc.tl.umap(adata)
sc.tl.embedding_density(adata,groupby='time')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:301: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:427: UserWarning: Exited at iteration 2000 with accuracies 
[1.90064632e-14 1.69639187e-07 5.23307462e-07 6.90563262e-06]
not reaching the requested tolerance 1.341104507446289e-06.
Use iteration 1196 instead with accuracy 
1.0973872583158677e-06.

  _, diffusion_map = lobpcg(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:427: UserWarning: Exited postprocessing with accuracies 
[1.60210635e-15 1.70558515e-07 5.26268960e-07 3.69250512e-06]
not reaching the requested tolerance 1.341104507446289e-06.
  _, diffusion_map = lobpcg(


In [40]:
adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')

/tmp/ipykernel_78831/2877064320.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')


In [45]:
kbet_score_cyto = kBET(
    adata[adata.obs.time.isin(['CS10','CS10-CytOrigin'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scde = kBET(
    adata[adata.obs.time.isin(['CS10','CS10-scDesign3'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scva = kBET(
    adata[adata.obs.time.isin(['CS10','CS10-scVAEDer'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_spla = kBET(
    adata[adata.obs.time.isin(['CS10','CS10-Splatter'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_simp = kBET(
    adata[adata.obs.time.isin(['CS10','CS10-Splatter-Simple'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pand

0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


Adding diffusion to step 4


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_uniqu

...

/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


1 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global convers

4 labels consist of a single batch or is too small. Skip.
4 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(


In [ ]:
dfs = {
    'CytOrigin': kbet_score_cyto,
    'scVAEDer': kbet_score_scva,
    'scDesign3': kbet_score_scde,
    'Splatter': kbet_score_spla,
    'Splatter-Simple': kbet_score_simp
}
combined = pd.concat([df.set_index('cluster')['kBET'].rename(name) for name, df in dfs.items()], axis=1)
combined.fillna(0, inplace=True)
combined.loc['mean'] = combined.mean(axis=0)
combined.T.to_csv('kBET/CS10_full_pca.csv')

# CS11

In [48]:
cyto = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS11f_CytO_new_new_clustering.h5ad')
spla = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS11f_splatter.h5ad')
simp = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS11f_splatter_simple.h5ad')
scde = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS11f_scDesign3.h5ad')
scva = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/benchmark/simulated_matrix/h5ad_format/CS11f_scVAEDer.h5ad')
real = sc.read_h5ad('/slurm/home/yrd/liaolab/caohaoxue/embryo_work/data_standard/CS11.h5ad')

In [49]:
adata_int = sc.concat([cyto,scde,scva,spla,simp],
                      join = 'outer',
                      fill_value = 0)

adata = sc.concat([real,adata_int],
                 label = 'source',
                 keys = ['real','simulated'],
                 index_unique = '-')

set_seed(seed=42)
sc.tl.pca(adata, svd_solver='arpack')
adata.obs['stage'] = 'CS11'
sc.external.pp.bbknn(adata, batch_key='source', pynndescent_random_state=42)
sc.tl.umap(adata)
sc.tl.embedding_density(adata,groupby='time')

/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/merge.py:1278: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


/slurm/home/yrd/liaolab/caohaoxue/.local/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:301: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [50]:
adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')

/tmp/ipykernel_78831/2877064320.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs['merge_type'] = adata.obs['merge_type'].replace(['Unknown'], 'Others')


In [55]:
kbet_score_cyto = kBET(
    adata[adata.obs.time.isin(['CS11','CS11-CytOrigin'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scde = kBET(
    adata[adata.obs.time.isin(['CS11','CS11-scDesign3'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_scva = kBET(
    adata[adata.obs.time.isin(['CS11','CS11-scVAEDer'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_spla = kBET(
    adata[adata.obs.time.isin(['CS11','CS11-Splatter'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)

kbet_score_simp = kBET(
    adata[adata.obs.time.isin(['CS11','CS11-Splatter-Simple'])],
    batch_key="time",
    label_key="merge_type",
    type_="full",return_df=True,
    embed="X_pca",scaled=False
)


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


0 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global convers

...

/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of o

4 labels consist of a single batch or is too small. Skip.
4 labels consist of a single batch or is too small. Skip.


/slurm/home/yrd/liaolab/caohaoxue/anaconda3/envs/scib_env/lib/python3.9/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_tmp.obs.groupby(label_key).agg(


In [ ]:
dfs = {
    'CytOrigin': kbet_score_cyto,
    'scVAEDer': kbet_score_scva,
    'scDesign3': kbet_score_scde,
    'Splatter': kbet_score_spla,
    'Splatter-Simple': kbet_score_simp
}
combined = pd.concat([df.set_index('cluster')['kBET'].rename(name) for name, df in dfs.items()], axis=1)
combined.fillna(0, inplace=True)
combined.loc['mean'] = combined.mean(axis=0)
combined.T.to_csv('kBET/CS11_full_pca.csv')